# Perform some SQLite database operations.

This snippet demonstrates how to interact with an SQLite 
database by establishing a connection, creating a cursor 
object, executing SQL commands using that cursor, and 
closing the connection.

In [1]:
import sqlite3

## Create, insert data into to the the db

In [2]:
# create a database if it doesn't exist
conn = sqlite3.connect('database1.db')

# create a cursor
cursor = conn.cursor()

# Create table
cursor.execute('CREATE TABLE users (name TEXT, age INTEGER)')

# Insert data
cursor.execute("INSERT INTO users VALUES ('John', 30)")

# Save (commit) the changes 
conn.commit()    

# Close the connection
conn.close()

## EXECUTEMANY

In [5]:
conn = sqlite3.connect('database1.db')
c = conn.cursor()

# Create table
c.execute('''CREATE TABLE stocks
             (date text, trans text, symbol text, qty real, price real)''')

# Data to be inserted (a list of tuples)
stocks = [
    ('2006-01-05', 'BUY', 'RHAT', 100, 35.14),
    ('2006-03-28', 'BUY', 'LUV', 50, 36.21),
    ('2006-04-05','SELL', 'IBM', 70, 76.47)  
] 

# Using executemany() to insert multiple rows    
c.executemany('INSERT INTO stocks VALUES (?,?,?,?,?)', stocks)

## Select data from db

In [12]:
# Create table
cursor.execute('SELECT * FROM stocks WHERE price = 35.14')

# Display data
for row in cursor:
    print(*row)

2006-01-05 BUY RHAT 100.0 35.14


## EXECUTESCRIPT

In [13]:
# the script can be in a seperate text file

sql_statements = '''
    CREATE TABLE customers (
        id INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
        first_name TEXT NOT NULL,
        last_name TEXT NOT NULL 
    );
    
    CREATE TABLE orders (
        id INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
        customer_id INTEGER NOT NULL,     
        product_name TEXT NOT NULL,
        FOREIGN KEY (customer_id) REFERENCES customers (id) 
    );
'''

In [53]:
# Execute the script
cursor.executescript(sql_statements)

ProgrammingError: Cannot operate on a closed database.

In [20]:
# Instanciate the table
customer_list = [
    ('John', 'Smith'),
    ('Jane', 'Doe'),
    ('Jack', 'Williams'),
    ('Jill', 'Brown'),
    ('Tom', 'Johnson'),
    ('Mary', 'Miller'),
    ('Kim', 'Davis'),
    ('Jim', 'Jones'),
    ('Lisa', 'Taylor'),
    ('Kate', 'Moore')
]
cursor.executemany('INSERT INTO customers (first_name, last_name) VALUES (?, ?)', customer_list)

In [37]:
cursor.execute('SELECT (first_name) FROM customers;')
for customer in cursor:
    print(*customer)

John
Jane
Jack
Jill
Tom
Mary
Kim
Jim
Lisa
Kate


In [22]:
cursor.execute('ALTER TABLE customers ADD COLUMN city VARCHAR;')

In [23]:
cursor.execute('ALTER TABLE customers ADD COLUMN country VARCHAR;')

In [44]:
# This doesn't work
cities1 = [
    ('John', 'Smith', 'London', 'United Kingdom'),   
    ('Jane', 'Doe', 'Paris', 'France'),   
    ('Jack', 'Williams','Tokyo', 'Japan'),   
    ('Jill', 'Brown','Beijing', 'China'),
    ('Tom', 'Johnson', 'New York', 'United States'),    
    ('Mary', 'Miller','Dubai', 'United Arab Emirates'),
    ('Kim', 'Davis', 'Moscow', 'Russia'),       
    ('Jim', 'Jones','Mumbai', 'India'),  
    ('Lisa', 'Taylor','Bangkok', 'Thailand'),   
    ('Kate', 'Moore', 'Lagos', 'Nigeria')  
]

# This worked
cities2 = [
    ('London', 'United Kingdom', 'John', 'Smith'),   
    ('Paris', 'France', 'Jane', 'Doe'),   
    ('Tokyo', 'Japan', 'Jack', 'Williams'),   
    ('Beijing', 'China','Jill', 'Brown'),
    ('New York', 'United States', 'Tom', 'Johnson'),    
    ('Dubai', 'United Arab Emirates', 'Mary', 'Miller'),
    ('Moscow', 'Russia', 'Kim', 'Davis'),       
    ('Mumbai', 'India','Jim', 'Jones'),  
    ('Bangkok', 'Thailand','Lisa', 'Taylor'),   
    ('Lagos', 'Nigeria', 'Kate', 'Moore')   
]

In [45]:
# The order of elements must be the same as in the statement
cursor.executemany('UPDATE customers SET city=?, country=? WHERE first_name=? AND last_name=?;',
                   cities2)

In [47]:
# Retrieving data
cursor.execute('SELECT * FROM customers;')
for custom in cursor:
    print(*custom)

1 John Smith London United Kingdom
2 Jane Doe Paris France
3 Jack Williams Tokyo Japan
4 Jill Brown Beijing China
5 Tom Johnson New York United States
6 Mary Miller Dubai United Arab Emirates
7 Kim Davis Moscow Russia
8 Jim Jones Mumbai India
9 Lisa Taylor Bangkok Thailand
10 Kate Moore Lagos Nigeria


## Closing the cursor and the connector

In [48]:
# Save (commit) the changes 
conn.commit()

# Close the connection
conn.close()

## ReOpen database

In [54]:
conn = sqlite3.connect('database1.db')
cursor = conn.cursor()

In [64]:
products = [
    (1, 'iPhone 11', 799), 
    (5, 'Macbook Pro', 1299),       
    (10,  'iPad', 329),    
    (8, 'Apple Watch', 399),       
    (2, 'Airpods', 159),     
    (6, 'Samsung Galaxy S20', 999),       
    (3, 'Sony Headphones', 129), 
    (9, 'Nikon Camera', 599),     
    (4, 'GoPro Hero', 199),      
    (7, 'Google Nest Hub', 129)
]

In [57]:
cursor.execute('SELECT * FROM orders')

In [58]:
cursor.execute('ALTER TABLE orders ADD price TEXT;')

In [65]:
cursor.executemany('INSERT INTO orders (customer_id, product_name, price) VALUES (?,?,?)', products)

In [68]:
cursor.execute('SELECT * FROM orders WHERE customer_id=5;')

In [69]:
for row in cursor:
    print(*row)

2 5 Macbook Pro 1299


## JONCTION ONE TO MANY

In [72]:
cursor.execute('SELECT * FROM orders \
                JOIN customers ON orders.customer_id = customers.id \
                WHERE customers.id = 1')

In [73]:
for row in cursor:
    print(*row)

1 1 iPhone 11 799 1 John Smith London United Kingdom
